# 🧹 Notebook 02: Data Preprocessing

**Objective:** Clean the merged dataset — handle missing values, outliers, and validate data quality.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config, load_csv
from src.data.data_cleaning import (
    report_missing, handle_missing_values,
    remove_outliers_iqr, validate_data_quality
)

config = load_config('../config/config.yaml')
df = load_csv('../' + config['data']['merged_path'])
print(f"Loaded shape: {df.shape}")
df.head()

## 2.1 Missing Value Analysis

In [ ]:
# Report missing values
missing_report = report_missing(df, 'Merged Data')

# Visualize missing pattern
if df.isnull().sum().sum() > 0:
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df.isnull().T, cbar=True, cmap='YlOrRd', ax=ax)
    ax.set_title('Missing Value Pattern')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values to visualize.')

## 2.2 Handle Missing Values

In [ ]:
# Apply linear interpolation (best for time series)
df_clean = handle_missing_values(df, method='interpolate', limit=6)

print(f"Before cleaning: {len(df)} rows")
print(f"After cleaning:  {len(df_clean)} rows")
print(f"Remaining missing: {df_clean.isnull().sum().sum()}")

## 2.3 Outlier Treatment

In [ ]:
# Box plots before outlier treatment
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
pollutant_cols = ['pm2_5', 'pm10', 'no2', 'so2', 'co', 'o3']
for ax, col in zip(axes.flat, pollutant_cols):
    if col in df_clean.columns:
        df_clean[col].plot.box(ax=ax)
        ax.set_title(col.upper())
plt.suptitle('Pollutant Distributions (Before Outlier Treatment)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Apply IQR-based outlier capping (conservative: factor=3.0)
df_clean = remove_outliers_iqr(df_clean, factor=3.0)

# Box plots after
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, pollutant_cols):
    if col in df_clean.columns:
        df_clean[col].plot.box(ax=ax)
        ax.set_title(col.upper())
plt.suptitle('Pollutant Distributions (After Outlier Treatment)', fontsize=14)
plt.tight_layout()
plt.show()

## 2.4 Data Quality Validation

In [ ]:
quality = validate_data_quality(df_clean, 'Cleaned Data')

# Save cleaned data
df_clean.to_csv('../' + config['data']['merged_path'])
print(f"\nCleaned data saved. Final shape: {df_clean.shape}")
df_clean.describe().round(2)

## ✅ Preprocessing Complete

**Next:** Notebook 03 — AQI Calculation